## 01 — Bronze: ingestion

Makes the data **technically** usable (typing, date parsing, normalization) with
no business logic; validity rules and joins happen in Silver.

**Input:** the CSV dropped by `00_generation` in the landing zone (immutable raw source).
**External sources:** public holidays ( from datagouv).
**Outputs:** `sncf_gc.bronze.frequentation`, `.jours_feries`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

LANDING_PATH = "/Volumes/sncf_gc/bronze/landing/frequentation.csv"  # same path as in 00

## Frequentation : technical cleaning
Reads the raw CSV (all columns as string) dropped by notebook 00, then types,
parses dates (two source formats) and normalizes casing.

In [0]:
def clean_frequentation(df):
    """Technical cleaning of the frequentation data (no business logic).

    Parses dates (two source formats), casts column types, and normalizes
    casing/whitespace. Exact deduplication is applied LAST, on normalized data,
    to catch duplicates that only differed by casing or date format.

    Args:
        df: raw DataFrame (string columns) read from the landing zone.
    Returns:
        Typed, normalized DataFrame with exact duplicates removed.
    """
    return (
        df
        .withColumn("date", F.to_date(F.coalesce(
            F.try_to_timestamp(F.col("date"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("date"), F.lit("dd/MM/yyyy")),
        )))
        .withColumn("gare_id", F.col("gare_id").cast(IntegerType()))
        .withColumn("heure_tranche", F.trim(F.col("heure_tranche")).cast(IntegerType()))
        .withColumn("nb_voyageurs", F.col("nb_voyageurs").cast(IntegerType()))
        .withColumn("nb_non_voyageurs", F.col("nb_non_voyageurs").cast(IntegerType()))
        .withColumn("latitude", F.col("latitude").cast(DoubleType()))
        .withColumn("longitude", F.col("longitude").cast(DoubleType()))
        .withColumn("region", F.initcap(F.trim(F.col("region"))))
        .withColumn("type_gare", F.initcap(F.trim(F.col("type_gare"))))
        .withColumn("ville", F.initcap(F.trim(F.col("ville"))))
        .withColumn("nom_gare", F.trim(F.col("nom_gare")))
        .withColumn("code_uic", F.trim(F.col("code_uic")))
        .dropDuplicates()  
    )


# picks up what 00 generated
df_raw = spark.read.option("header", True).csv(LANDING_PATH)

df_bronze = (
    clean_frequentation(df_raw)
    .withColumn("ingestion_timestamp", F.current_timestamp())  # outside the function: non-deterministic
)

df_bronze.write.format("delta").mode("overwrite").saveAsTable("sncf_gc.bronze.frequentation")
print(f"✅ bronze.frequentation: {spark.table('sncf_gc.bronze.frequentation').count()} rows")

In [0]:
def load_public_holidays():
    """Read French public holidays from the CSV dropped in the landing zone.

    Source file downloaded once from datagouv.
    Returns:
        DataFrame [date: date, nom_jour_ferie: string].
    """
    return (
        spark.read.option("header", True)
        .csv("/Volumes/sncf_gc/bronze/landing/jours_feries.csv")
        .select(
            F.to_date("date").alias("date"),
            F.col("nom_jour_ferie"),
        ).dropDuplicates(["date"])
    )


(load_public_holidays()
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .write.format("delta").mode("overwrite").saveAsTable("sncf_gc.bronze.jours_feries"))
print("✅ bronze.jours_feries loaded")

In [0]:
def load_commune_population():
    """Commune population (latest INSEE figures), hardcoded.

    Returns:
        DataFrame [nom_commune: string, code_insee: string, population: int].
    """
    data = [
        ("Paris",       "75056", 2133111),
        ("Marseille",   "13055", 873076),
        ("Lyon",        "69123", 522969),
        ("Toulouse",    "31555", 504078),
        ("Bordeaux",    "33063", 261804),
        ("Lille",       "59350", 236710),
        ("Strasbourg",  "67482", 290576),
        ("Nantes",      "44109", 320732),
        ("Rennes",      "35238", 221272),
        ("Nice",        "06088", 348085),
        ("Montpellier", "34172", 299096),
        ("Le Havre",    "76351", 165830),
        ("Dijon",       "21231", 159346),
        ("Reims",       "51454", 181194),
        ("Tours",       "37261", 137658),
        ("Grenoble",    "38185", 156389),
        ("Angers",      "49007", 155850),
    ]
    return spark.createDataFrame(data, ["nom_commune", "code_insee", "population"])


(load_commune_population()
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .write.format("delta").mode("overwrite").saveAsTable("sncf_gc.bronze.population_communes"))
print("✅ bronze.population_communes loaded")

In [0]:
# ── 1. Frequentation: schema, volume, typing ──

def test_bronze_frequentation():
    """Schema contract, non-empty volume, successful technical typing."""
    bronze = spark.table("sncf_gc.bronze.frequentation")
    expected = {"gare_id", "code_uic", "nom_gare", "ville", "region", "type_gare",
                "segment", "latitude", "longitude", "date", "heure_tranche",
                "nb_voyageurs", "nb_non_voyageurs", "ingestion_timestamp"}
    
    assert not (expected - set(bronze.columns)), " missing columns"
    assert bronze.count() > 0, " empty table"
    dtypes = dict(bronze.dtypes)
    assert dtypes["date"] == "date", "date not typed"
    assert dtypes["nb_voyageurs"] == "int", " nb_voyageurs not typed"
    print("✅ test_bronze_frequentation OK")

# ── 2. External sources: availability + unique join keys ──
def test_bronze_jours_feries():
    """The `date` key must be unique, otherwise the Silver join fans out."""
    f = spark.table("sncf_gc.bronze.jours_feries")
    assert f.count() > 0, " empty"
    assert f.count() == f.select("date").distinct().count(), " date not unique "
    print("✅ test_bronze_jours_feries OK")


def test_bronze_population():
    """Population reference loaded with a unique commune key."""
    p = spark.table("sncf_gc.bronze.population_communes")
    assert p.count() > 0, " empty"
    assert p.count() == p.select("nom_commune").distinct().count(), " nom_commune not unique"
    print("✅ test_bronze_population OK")


test_bronze_frequentation()
test_bronze_jours_feries()
test_bronze_population()

# ── 3. Data quality monitoring
raw_count = spark.read.option("header", True).csv(LANDING_PATH).count()
bronze = spark.table("sncf_gc.bronze.frequentation")
bronze_count = bronze.count()
print(f"\nRows raw: {raw_count} | after dedup: {bronze_count} | removed: {raw_count - bronze_count}")

total = bronze_count
for c in ["date", "gare_id", "nb_voyageurs"]:
    nulls = bronze.filter(F.col(c).isNull()).count()
    print(f"  null rate {c}: {nulls/total:.1%}")